In [1]:
pip install selenium

     ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
     ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
     ---------------------------------------- 0.0/9.5 MB 445.2 kB/s eta 0:00:22
     ---------------------------------------- 0.0/9.5 MB 445.2 kB/s eta 0:00:22
     ---------------------------------------- 0.0/9.5 MB 445.2 kB/s eta 0:00:22
     ---------------------------------------- 0.0/9.5 MB 164.3 kB/s eta 0:00:58
     ---------------------------------------- 0.1/9.5 MB 286.7 kB/s eta 0:00:33
     ---------------------------------------- 0.1/9.5 MB 328.2 kB/s eta 0:00:29
     ---------------------------------------- 0.1/9.5 MB 328.2 kB/s eta 0:00:29
      --------------------------------------- 0.1/9.5 MB 288.8 kB/s eta 0:00:33
      --------------------------------------- 0.2/9.5 MB 430.1 kB/s eta 0:00:22
     - -------------------------------------- 0.2/9.5 MB 457.3 kB/s eta 0:00:21
     - -------------------------------------- 0.2/9.5 MB 4


[notice] A new release of pip is available: 23.0.1 -> 25.0.1
[notice] To update, run: C:\Users\15520\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
import time
path = r"D:\BigData\chromedriver-win64\chromedriver-win64\chromedriver.exe"

ser = Service(path)
browser = webdriver.Chrome(service= ser)
browser.get("https://batdongsan.com.vn/nha-dat-ban")
browser.maximize_window()
time.sleep(2)

In [2]:
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [23]:
!pip install pandas


[notice] A new release of pip is available: 23.0.1 -> 25.0.1
[notice] To update, run: C:\Users\15520\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [6]:
import pandas as pd
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Khởi tạo trình duyệt
browser = webdriver.Chrome()

# Danh sách chứa dữ liệu
data_list = []

# Số trang tối đa cần quét
max_pages = 5  # Giới hạn để tránh bị chặn
base_url = "https://batdongsan.com.vn/nha-dat-ban/p{}"

for page in range(1, max_pages + 1):
    url = base_url.format(page)
    print(f"Đang thu thập dữ liệu trang {page}: {url}")
    browser.get(url)
    
    try:
        # Đợi các thẻ cha chứa toàn bộ thông tin bất động sản hiển thị
        eles = WebDriverWait(browser, 10).until(
            EC.presence_of_all_elements_located((By.CLASS_NAME, "js__card"))
        )
    except:
        print(f"Không thể tải trang {page}, dừng lại!")
        break  # Chỉ có trong vòng lặp for, KHÔNG để ngoài cùng

    for ele in eles:
        try:
            # Lấy tiêu đề
            name_project = ele.find_element(By.XPATH, './/span[contains(@class, "pr-title")]').text
        except:
            name_project = "Không tìm thấy tên dự án"

        try:
            # Lấy giá
            price_project = ele.find_element(By.XPATH, './/span[contains(@class, "re__card-config-price")]').text
        except:
            price_project = "Không tìm thấy giá"

        try:
            # Lấy diện tích
            area_project = ele.find_element(By.XPATH, './/span[contains(@class, "re__card-config-area")]').text
        except:
            area_project = "Không tìm thấy diện tích"

        try:
            bedrooms = ele.find_element(By.XPATH, './/span[contains(@class, "re__card-config-bedroom")]').text
        except:
            bedrooms = "Không có thông tin phòng ngủ"

        try:
            toilets = ele.find_element(By.XPATH, './/span[contains(@class, "re__card-config-toilet")]').text
        except:
            toilets = "Không có thông tin WC"

        try:
            location = ele.find_element(By.XPATH, './/div[contains(@class, "re__card-location")]/span[2]').text
        except:
            location = "Không có thông tin vị trí"

        try:
            # Lấy URL chi tiết từ thẻ <a>
            detail_url = ele.find_element(By.XPATH, './/a[contains(@class, "js__product-link-for-product-id")]').get_attribute("href")
        except:
            detail_url = None

        # ---- MỞ TRANG CHI TIẾT ĐỂ LẤY ĐỊA CHỈ ----
        if detail_url:
            browser.execute_script("window.open(arguments[0]);", detail_url)  # Mở link trong tab mới
            browser.switch_to.window(browser.window_handles[1])  # Chuyển sang tab mới
            time.sleep(2)  # Đợi trang load

            try:
                # Lấy địa chỉ cụ thể
                address = WebDriverWait(browser, 5).until(
                    EC.presence_of_element_located((By.CLASS_NAME, "js__pr-address"))
                ).text
            except:
                address = "Không có địa chỉ chi tiết"

            browser.close()  # Đóng tab chi tiết
            browser.switch_to.window(browser.window_handles[0])  # Quay lại tab chính
        else:
            address = "Không có địa chỉ chi tiết"

        # Thêm dữ liệu vào danh sách
        data_list.append([name_project, price_project, area_project,bedrooms,toilets,location,address, detail_url])

    # Đợi 2 giây để tránh bị chặn
    time.sleep(2)

# Lưu dữ liệu vào CSV
df = pd.DataFrame(data_list, columns=["Tên dự án", "Giá", "Diện tích","Phòng ngủ","Phòng vệ sinh","Vị trí","Địa chỉ chi tiết","URL chi tiết"])
df.to_csv(r"D:\bat_dong_san.csv", index=False, encoding="utf-8-sig")

print("Hoàn thành! Dữ liệu đã được lưu vào D:\\bat_dong_san.csv")
browser.quit()


Đang thu thập dữ liệu trang 1: https://batdongsan.com.vn/nha-dat-ban/p1
Đang thu thập dữ liệu trang 2: https://batdongsan.com.vn/nha-dat-ban/p2
Không thể tải trang 2, dừng lại!
Hoàn thành! Dữ liệu đã được lưu vào D:\bat_dong_san.csv


 OpenStreetMap (OSM) để lấy tọa độ và tìm các địa điểm xung quanh

In [1]:
!pip install requests folium

     ---------------------------------------- 0.0/64.9 kB ? eta -:--:--
     ------------------ --------------------- 30.7/64.9 kB 1.4 MB/s eta 0:00:01
     ----------------------------------- -- 61.4/64.9 kB 656.4 kB/s eta 0:00:01
     -------------------------------------- 64.9/64.9 kB 588.0 kB/s eta 0:00:00
     ---------------------------------------- 0.0/110.9 kB ? eta -:--:--
     -------------- ------------------------- 41.0/110.9 kB ? eta -:--:--
     ----------------------------------- -- 102.4/110.9 kB 1.2 MB/s eta 0:00:01
     -------------------------------------- 110.9/110.9 kB 1.1 MB/s eta 0:00:00
     ---------------------------------------- 0.0/102.8 kB ? eta -:--:--
     ----------- --------------------------- 30.7/102.8 kB 1.3 MB/s eta 0:00:01
     ----------------------------- ------- 81.9/102.8 kB 919.0 kB/s eta 0:00:01
     ------------------------------------ 102.8/102.8 kB 984.6 kB/s eta 0:00:00
     ---------------------------------------- 0.0/134.9 kB ? eta -:-


[notice] A new release of pip is available: 23.0.1 -> 25.0.1
[notice] To update, run: C:\Users\15520\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [11]:
import requests

def get_coordinates_osm(address):
    url = f"https://nominatim.openstreetmap.org/search?q={address}&format=json"
    headers = {
        "User-Agent": "MyGeocodingApp/1.0 (22521658@gm.uit.edu.vn)"
    }

    response = requests.get(url, headers=headers)

    if response.status_code == 403:
        print("❌ Bị chặn! Hãy thử đổi mạng hoặc dùng API khác.")
        return None, None

    data = response.json()
    if data:
        lat = float(data[0]["lat"])
        lon = float(data[0]["lon"])
        return lat, lon
    return None, None

# Thử lại với User-Agent mới
#address = "Xã Tân Hội, Đan Phượng, Hà Nội"
address = "Phường Phú Hữu, Quận 9, Hồ Chí Minh"

latitude, longitude = get_coordinates_osm(address)

print(f"📍 Tọa độ: {latitude}, {longitude}")


📍 Tọa độ: 10.7608529, 106.7012374


In [ ]:
import requests
from math import radians, cos, sin, sqrt, atan2

# Hàm tính khoảng cách giữa hai tọa độ (Haversine formula)
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Bán kính Trái Đất (km)
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))
    
    return R * c  # Khoảng cách (km)

# Hàm tìm các địa điểm xung quanh
def find_nearby_places(lat, lon, radius=5000):
    overpass_url = "http://overpass-api.de/api/interpreter"
    overpass_query = f"""
    [out:json];
    (
      node(around:{radius},{lat},{lon})["amenity"="hospital"];    // Bệnh viện
      node(around:{radius},{lat},{lon})["amenity"="school"];      // Trường học
      node(around:{radius},{lat},{lon})["amenity"="marketplace"]; // Chợ
      node(around:{radius},{lat},{lon})["shop"="mall"];           // Trung tâm thương mại
    );
    out body;
    """

    headers = {"User-Agent": "MyOSMApp/1.0 (your_email@example.com)"}
    response = requests.get(overpass_url, params={"data": overpass_query}, headers=headers)

    if response.status_code != 200:
        print("❌ Không thể truy vấn dữ liệu, thử lại sau!")
        return []

    data = response.json()
    places = []
    for element in data.get("elements", []):
        name = element.get("tags", {}).get("name", "Không có tên")
        place_lat = element["lat"]
        place_lon = element["lon"]
        distance = haversine(lat, lon, place_lat, place_lon)  # Tính khoảng cách
        places.append((name, distance))

    return sorted(places, key=lambda x: x[1])  # Sắp xếp theo khoảng cách

# Chạy hàm với tọa độ của bạn
latitude = 21.0966838
longitude = 105.86806209756409

results = find_nearby_places(latitude, longitude)
if results:
    for place in results:
        print(f"📍 {place[0]} - Khoảng cách: {place[1]:.2f} km")
else:
    print("⚠️ Không tìm thấy địa điểm nào trong phạm vi 5km.")


📍 Trạm y tế xã Xuân Canh - Khoảng cách: 1.69 km
📍 Không có tên - Khoảng cách: 1.74 km
📍 Không có tên - Khoảng cách: 1.88 km
📍 Không có tên - Khoảng cách: 3.98 km
📍 Trường Trung cấp Y - Dược Cộng đồng Hà Nội - Khoảng cách: 4.03 km
📍 Không có tên - Khoảng cách: 4.85 km
📍 Trường Tiểu Học Ngô Gia Tự - Khoảng cách: 4.95 km
📍 Trường THCS Ngô Gia Tự - Khoảng cách: 4.97 km


In [3]:
import requests

def reverse_geocode(lat, lon):
    url = f"https://nominatim.openstreetmap.org/reverse?lat={lat}&lon={lon}&format=json"
    headers = {
        "User-Agent": "MyReverseGeocodeApp/1.0 (your_email@example.com)"
    }

    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        print("❌ Không thể lấy dữ liệu, thử lại sau!")
        return None

    data = response.json()
    address = data.get("display_name", "Không tìm thấy địa chỉ")
    return address

# Thử với tọa độ đã có
latitude = 21.09337
longitude = 105.6980589

address = reverse_geocode(latitude, longitude)
print(f"📍 Địa chỉ: {address}")


❌ Không thể lấy dữ liệu, thử lại sau!
📍 Địa chỉ: None


XỬ LÝ CỘT ĐỊA CHỈ

In [7]:
import csv

# Hàm xử lý chuỗi, lấy phần sau dấu phẩy đầu tiên
def extract_after_first_comma(text):
    parts = text.split(",", 1)
    return parts[1].strip() if len(parts) > 1 else text

# Hàm xử lý file CSV dành cho cột "Địa chỉ chi tiết"
def process_csv(input_file, output_file):
    with open(input_file, mode='r', encoding='utf-8') as infile, open(output_file, mode='w', encoding='utf-8', newline='') as outfile:
        reader = csv.reader(infile)
        writer = csv.writer(outfile)

        header = next(reader)  # Đọc tiêu đề cột
        if "Địa chỉ chi tiết" not in header:
            print("❌ Không tìm thấy cột 'Địa chỉ chi tiết' trong file CSV!")
            return

        # Tìm vị trí cột "Địa chỉ chi tiết"
        address_index = header.index("Địa chỉ chi tiết")
        writer.writerow(header)  # Ghi tiêu đề vào file mới

        for row in reader:
            if len(row) > address_index:  # Kiểm tra xem dòng có đủ cột không
                row[address_index] = extract_after_first_comma(row[address_index])  # Chỉ xử lý cột này
                writer.writerow(row)

# Sử dụng hàm
input_csv = "batdongsan.csv"   # File CSV đầu vào
output_csv = "batdongsan_replace.csv" # File CSV sau xử lý
process_csv(input_csv, output_csv)
print(f"✅ File đã được xử lý và lưu vào {output_csv}")


✅ File đã được xử lý và lưu vào batdongsan_replace.csv


In [14]:
import requests
import csv
import time

# Hàm lấy tọa độ từ địa chỉ
def get_coordinates_osm(address):
    url = f"https://nominatim.openstreetmap.org/search?q={address}&format=json"
    headers = {
        "User-Agent": "MyGeocodingApp/1.0 (22521658@gm.uit.edu.vn)"
    }

    response = requests.get(url, headers=headers)

    if response.status_code == 403:
        print(f"❌ Bị chặn khi tìm {address}! Hãy thử đổi mạng hoặc dùng API khác.")
        return None, None

    data = response.json()
    if data:
        lat = float(data[0]["lat"])
        lon = float(data[0]["lon"])
        return lat, lon
    return None, None

# Hàm xử lý file CSV
def process_csv(input_file, output_file):
    with open(input_file, mode='r', encoding='utf-8') as infile, open(output_file, mode='w', encoding='utf-8', newline='') as outfile:
        reader = csv.DictReader(infile)
        fieldnames = reader.fieldnames + ["Latitude", "Longitude"]  # Thêm cột mới
        writer = csv.DictWriter(outfile, fieldnames=fieldnames)
        
        writer.writeheader()  # Ghi tiêu đề
        
        for row in reader:
            address = row["Địa chỉ chi tiết"]
            print(f"🔍 Đang tìm tọa độ cho: {address}")
            lat, lon = get_coordinates_osm(address)
            row["Latitude"] = lat
            row["Longitude"] = lon
            writer.writerow(row)

            time.sleep(1)  # Tránh bị chặn do gửi quá nhiều request

# Sử dụng hàm
input_csv = "batdongsan_replace.csv"   # Thay bằng tên file đầu vào
output_csv = "batdongsanfinal.csv" # File đầu ra có thêm tọa độ

process_csv(input_csv, output_csv)
print(f"✅ File đã được xử lý và lưu vào {output_csv}")


🔍 Đang tìm tọa độ cho: Xã Tân Hội, Đan Phượng, Hà Nội
🔍 Đang tìm tọa độ cho: Xã Tân Hội, Đan Phượng, Hà Nội
🔍 Đang tìm tọa độ cho: Hữu Nghị, Phù Chẩn, Từ Sơn, Bắc Ninh
🔍 Đang tìm tọa độ cho: Xã Tân Hội, Đan Phượng, Hà Nội
🔍 Đang tìm tọa độ cho: Phường Phú Thượng, Tây Hồ, Hà Nội
🔍 Đang tìm tọa độ cho: Mai Chí Thọ, Phường An Phú, Quận 2, Hồ Chí Minh
🔍 Đang tìm tọa độ cho: Cổ Loa, Xã Đông Hội, Đông Anh, Hà Nội
🔍 Đang tìm tọa độ cho:  Trần Hưng Đạo, Phường Nại Hiên Đông , Sơn Trà, Đà Nẵng
🔍 Đang tìm tọa độ cho: Xã Tân Hội, Đan Phượng, Hà Nội
🔍 Đang tìm tọa độ cho: Trần Hưng Đạo, Phường Nại Hiên Đông , Sơn Trà, Đà Nẵng
🔍 Đang tìm tọa độ cho: Trần Hưng Đạo, Phường Nại Hiên Đông , Sơn Trà, Đà Nẵng
🔍 Đang tìm tọa độ cho:  Cổ Loa, Xã Đông Hội, Đông Anh, Hà Nội
🔍 Đang tìm tọa độ cho:  Xa Lộ Hà Nội, Phường Thảo Điền, Quận 2, Hồ Chí Minh
🔍 Đang tìm tọa độ cho:  Liên Phường, Phường Phú Hữu, Quận 9, Hồ Chí Minh
🔍 Đang tìm tọa độ cho:  Trần Thị Lý, Phường Mỹ An, Ngũ Hành Sơn, Đà Nẵng
🔍 Đang tìm tọa đ

In [2]:
!pip install fake_useragent

     ---------------------------------------- 0.0/125.8 kB ? eta -:--:--
     -------------------------------------- 125.8/125.8 kB 3.6 MB/s eta 0:00:00



[notice] A new release of pip is available: 23.0.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import undetected_chromedriver as uc
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium_stealth import stealth
import time
import csv
import random
from datetime import datetime
from fake_useragent import UserAgent

# Tạo tên file dựa trên thời gian hiện tại
current_time = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_filename = f"batdongsan_data_{current_time}.csv"

# Tự động cài đặt ChromeDriver phù hợp
service = Service(ChromeDriverManager().install())

# Tạo User-Agent ngẫu nhiên
ua = UserAgent()

# Cấu hình trình duyệt với undetected_chromedriver
options = uc.ChromeOptions()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--window-size=1920,1080")
options.add_argument("--disable-popup-blocking")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--disable-extensions")
options.add_argument("--disable-notifications")
options.add_argument(f"user-agent={ua.random}")


# Sử dụng undetected_chromedriver với ChromeDriver đã cập nhật
browser = uc.Chrome(
    service=service,
    options=options,
    use_subprocess=True,  # Thêm tùy chọn này để tránh lỗi
    version_main=134  # Cập nhật version Chrome phù hợp
)

# Cài đặt stealth mode
stealth(browser,
        languages=["en-US", "en"],
        vendor="Google Inc.",
        platform="Win32",
        webgl_vendor="Intel Inc.",
        renderer="Intel Iris OpenGL Engine",
        fix_hairline=True,
        hide_webdriver=True)

# Hàm kiểm tra Cloudflare
def check_cloudflare(driver):
    try:
        WebDriverWait(driver, 10).until(
            EC.title_contains("Just a moment")
        )
        print("⚠️ Phát hiện Cloudflare challenge...")
        return True
    except:
        return False

# Hàm xử lý Cloudflare
def handle_cloudflare(driver):
    print("🛡️ Đang xử lý Cloudflare...")
    time.sleep(10)  # Tăng thời gian chờ cho Cloudflare
    try:
        # Thử click vào nếu có button verify
        driver.find_element(By.XPATH, "//input[@type='checkbox']").click()
        time.sleep(5)
    except:
        pass
    try:
        driver.find_element(By.XPATH, "//input[@value='Verify']").click()
        time.sleep(5)
    except:
        pass

# Mở file CSV với tên tự động
with open(csv_filename, "w", newline="", encoding="utf-8-sig") as csvfile:
    writer = csv.writer(csvfile)
    # Đảm bảo đủ các cột như ban đầu
    writer.writerow(["Tên dự án", "Giá", "Diện tích", "Phòng ngủ", "Phòng vệ sinh", "Vị trí", "Địa chỉ chi tiết", "id"])

    # Duyệt qua từng trang
    for page in range(826, 831):  # Giảm số trang để test
        url = f"https://batdongsan.com.vn/nha-dat-ban/p{page}"
        print(f"📄 Đang xử lý trang: {url}")
        
        try:
            browser.get(url)
            
            # Kiểm tra Cloudflare
            if check_cloudflare(browser):
                handle_cloudflare(browser)
                if check_cloudflare(browser):  # Kiểm tra lại sau khi xử lý
                    print("❌ Không thể vượt qua Cloudflare. Bỏ qua trang này.")
                    continue
            
            # Chờ tải trang
            WebDriverWait(browser, 20).until(
                EC.presence_of_element_located((By.CLASS_NAME, "re__card-info-content"))
            )
            
            # Cuộn trang để tải dữ liệu
            for _ in range(5):
                browser.execute_script("window.scrollTo(0, document.body.scrollHeight);")
                time.sleep(random.uniform(2, 4))
            
            # Tìm danh sách bất động sản
            elements = browser.find_elements(By.CLASS_NAME, "re__card-info-content")
            if not elements:
                print(f"⚠️ Không tìm thấy dữ liệu trên trang {page}.")
                continue

            for ele in elements:
                try:
                    # Lấy thông tin cơ bản
                    name = ele.find_element(By.CLASS_NAME, "pr-title").text
                    price = ele.find_element(By.CLASS_NAME, "re__card-config-price").text
                    area = ele.find_element(By.CLASS_NAME, "re__card-config-area").text
                    
                    bedrooms = ele.find_element(By.CSS_SELECTOR, "span[class*='re__card-config-bedroom']").get_attribute("aria-label").split()[0] if ele.find_elements(By.CSS_SELECTOR, "span[class*='re__card-config-bedroom']") else "N/A"
                    bathrooms = ele.find_element(By.CSS_SELECTOR, "span[class*='re__card-config-toilet']").get_attribute("aria-label").split()[0] if ele.find_elements(By.CSS_SELECTOR, "span[class*='re__card-config-toilet']") else "N/A"
                    
                    location = ele.find_element(By.CSS_SELECTOR, "div[class*='re__card-location'] > span:last-child").text if ele.find_elements(By.CSS_SELECTOR, "div[class*='re__card-location'] > span:last-child") else "N/A"
                    detail_link = ele.find_element(By.XPATH, "./ancestor::a").get_attribute("href")

                

                    # Mở trang chi tiết trong tab mới
                    browser.execute_script(f"window.open('{detail_link}', '_blank');")
                    browser.switch_to.window(browser.window_handles[1])
                    
                    # Chờ và xử lý Cloudflare trên tab mới nếu có
                    time.sleep(random.uniform(3, 6))
                    if check_cloudflare(browser):
                        handle_cloudflare(browser)
                    
                    # Lấy thông tin chi tiết
                    try:
                        WebDriverWait(browser, 15).until(
                            EC.presence_of_element_located((By.CLASS_NAME, "re__pr-short-description"))
                        )
                        description = browser.find_element(By.XPATH, '//span[@class="re__pr-short-description js__pr-address"]').text
                    except:
                        description = "N/A"

                    # Lấy prid từ trang chi tiết
                    try:
                        prid_element = browser.find_element(By.XPATH, "//div[@id='product-detail-web']")
                        prid = prid_element.get_attribute("prid") if prid_element else "N/A"
                    except:
                        prid = "N/A"

                  
                    # Đóng tab chi tiết và quay lại trang danh sách
                    browser.close()
                    browser.switch_to.window(browser.window_handles[0])
                    
                    # Lưu vào CSV với đầy đủ thông tin
                    writer.writerow([
                        name, 
                        price, 
                        area, 
                        bedrooms, 
                        bathrooms, 
                        location, 
                        description,
                        prid,
                      
                    ])
                    
                    print(f"Trang {page}: {name} | {price} | {area} | {bedrooms} | {bathrooms} | {location} | {description} | {prid}")
                    # Ngủ ngẫu nhiên giữa các request
                    time.sleep(random.uniform(2, 5))
                    
                except Exception as e:
                    print(f"⚠️ Lỗi khi thu thập thông tin: {str(e)}")
                    # Đảm bảo đóng tab nếu có lỗi
                    if len(browser.window_handles) > 1:
                        browser.close()
                        browser.switch_to.window(browser.window_handles[0])
                    continue
                    
        except Exception as e:
            print(f"⚠️ Lỗi khi xử lý trang {page}: {str(e)}")
            continue

# Đóng trình duyệt
browser.quit()
print(f"✅ Dữ liệu đã được lưu vào file: {csv_filename}")

📄 Đang xử lý trang: https://batdongsan.com.vn/nha-dat-ban/p826
Trang 826: Chuyên CH Masteri Thảo Điền - cam kết báo giá thật - bao giá thấp nhất thị trường - hỗ trợ vay 80% | Giá thỏa thuận | 74 m² | 2 | 2 | Quận 2, Hồ Chí Minh | Dự án Masteri Thảo Điền, Đường Xa Lộ Hà Nội, Phường Thảo Điền, Quận 2, Hồ Chí Minh | 38623445
Trang 826: Cần bán mảnh đất 423m2 gần chợ Đông Tảo mặt tiền 24m giá đầu tư, LH 0961 694 *** | 8,2 tỷ | 423 m² | N/A | N/A | Khoái Châu, Hưng Yên | Xã Đông Tảo, Khoái Châu, Hưng Yên | 42531575
Trang 826: Nền góc sổ cá nhân ngay chung cư D1, DT: 308.2m2 (12x20m) đường 10m, khu đông đúc, sầm uất | 12,95 tỷ | 308,2 m² | N/A | N/A | Quận 8, Hồ Chí Minh | Đường Phạm Thế Hiển, Phường 7, Quận 8, Hồ Chí Minh | 40728984
Trang 826: Bán nhanh nửa sàn căn hộ Sài Gòn Pearl Sapphire 1 căn lớn thông tường 4 căn, DT 480m2 | 33 tỷ | 480 m² | N/A | N/A | Bình Thạnh, Hồ Chí Minh | Saigon Pearl, 92, Đường Nguyễn Hữu Cảnh, Phường 22, Bình Thạnh, Hồ Chí Minh | 39600782
Trang 826: Bán nhà ná